# Posos — arbitre Qwen2.5-Coder-14B sur les questions hésitantes du test

**GPU T4** (Exécution → Modifier le type d'exécution). Puis « Tout exécuter ».
Quand le bouton apparaît, dépose les 5 fichiers :
`input_train.csv`, `output_train.csv`, `input_test.csv` (dans `posos_camembert/data/`),
`ens5_qwen_w2_probs.npy` (dans `posos_camembert/submissions/`),
`intentions.json` (dans `posos_camembert/agent/`).

Sortie : `submission_arbitre.csv` (téléchargé à la fin).

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null   # l'installeur Ollama a besoin de zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama
!pip -q install ollama
import subprocess, time
serveur = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5-coder:14b 2>&1 | tail -1

In [ ]:
from google.colab import files
import os
for nom in files.upload():
    print(nom, os.path.getsize(nom))

In [ ]:
# Même logique que agent/arbitre.py (validée : 54 réponses corrigées / 16 cassées sur 720 cas)
import json, numpy as np, pandas as pd, ollama

x = pd.read_csv("input_train.csv"); y = pd.read_csv("output_train.csv")
df = x.merge(y, on="ID"); df["question"] = df["question"].fillna("").astype(str)
test = pd.read_csv("input_test.csv")
NOMS = json.load(open("intentions.json"))
P = np.load("ens5_qwen_w2_probs.npy")
MARGE = 0.15

def fiche(i):
    ex = df[df["intention"] == i]["question"]
    ex = ex[ex.str.len().between(20, 120)].sample(3, random_state=0)
    return (f"Intention {i} — {NOMS[str(i)]['nom']} : {NOMS[str(i)]['description']}\n"
            + "\n".join(f"  ex. : {q}" for q in ex))

def arbitrer(question, a, b):
    prompt = f"""Des questions sur les médicaments sont classées par intention (le type
d'information attendue). Pour la question ci-dessous, choisis la bonne intention entre
les deux candidates.

QUESTION : « {question[:400]} »

{fiche(a)}

{fiche(b)}

Réponds en JSON : {{"raison": "une phrase", "intention": {a} ou {b}}}"""
    r = ollama.chat(model="qwen2.5-coder:14b", format="json",
                    messages=[{"role": "user", "content": prompt}],
                    options={"temperature": 0, "num_ctx": 4096})
    try:
        choix = int(json.loads(r["message"]["content"])["intention"])
    except (ValueError, KeyError, TypeError, json.JSONDecodeError):
        return a
    return choix if choix in (a, b) else a

ordre = np.argsort(-P, axis=1)
lignes = np.arange(len(P))
idx = np.where(P[lignes, ordre[:, 0]] - P[lignes, ordre[:, 1]] < MARGE)[0]
pred = ordre[:, 0].copy()
journal = []
for n, k in enumerate(idx, 1):
    a, b = int(ordre[k, 0]), int(ordre[k, 1])
    choix = arbitrer(str(test["question"].iloc[k]), a, b)
    pred[k] = choix
    journal.append({"ID": int(test["ID"].iloc[k]), "ensemble": a, "autre": b, "qwen": choix})
    if n % 20 == 0:
        print(f"{n}/{len(idx)} — {sum(j['qwen'] != j['ensemble'] for j in journal)} changements")
changes = sum(j["qwen"] != j["ensemble"] for j in journal)
print(f"{len(idx)} questions hésitantes, {changes} réponses changées")

In [ ]:
pd.DataFrame({"ID": test["ID"], "intention": pred}).to_csv("submission_arbitre.csv", index=False)
pd.DataFrame(journal).to_csv("journal_arbitre.csv", index=False)
files.download("submission_arbitre.csv")
files.download("journal_arbitre.csv")